[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gonzalezulises/formacion-docente-bi-faces/blob/main/modulo-02-data-wrangling/notebooks/02_04_wrangling_avanzado.ipynb)

# 02.04 — Wrangling avanzado: limpieza, transformación y calidad de datos

**Módulo 2: Data Wrangling accesible (Python + SQL)**  
**Duración estimada:** 1.5 horas  
**Instructor:** Gabriel Gómez

---

## Objetivos

1. Leer datos desde archivos Excel con `read_excel`
2. Inspeccionar datasets con `.info()`, `.describe()`, `.dtypes`
3. Ordenar datos con `sort_values` y `sort_index`
4. Crear variables derivadas
5. Tratar datos faltantes con criterio contextual
6. Detectar y manejar outliers

In [ ]:
import pandas as pd
import numpy as np

# Configuración de visualización
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)

---

## 1. Lectura de datos desde Excel

En el mundo real, muchos datos llegan en archivos Excel. Pandas los lee directamente.

In [ ]:
# Lectura básica de Excel
# df = pd.read_excel('archivo.xlsx')

# Con opciones comunes
# df = pd.read_excel(
#     'presupuesto_2024.xlsx',
#     sheet_name='Enero',        # Hoja específica
#     skiprows=2,                 # Saltar filas de encabezado
#     usecols='A:F',             # Solo columnas A a F
#     dtype={'codigo': str}      # Forzar tipo de columna
# )

# Para este notebook, trabajamos con CSVs del repositorio
df_nomina = pd.read_csv('../datasets/empresarial/rrhh_nomina.csv')
df_nomina.head()

### Diferencias entre `read_csv` y `read_excel`

| Aspecto | `read_csv` | `read_excel` |
|---------|-----------|-------------|
| Velocidad | Rápido | Más lento |
| Dependencia | Ninguna extra | Requiere `openpyxl` |
| Hojas múltiples | No aplica | `sheet_name=` |
| Formato de fechas | Pueden ser strings | Suelen ser datetime |

```python
# Si necesitas openpyxl en Colab:
# !pip install openpyxl
```

---

## 2. Inspección profunda con `.info()`

`.info()` es tu primera herramienta de diagnóstico. Te dice qué tiene el dataset y qué problemas potenciales hay.

In [ ]:
df_rendimiento = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')
df_rendimiento.info()

In [ ]:
# ¿Qué buscar en .info()?
# 1. Non-Null Count: ¿hay columnas con menos valores que filas? → datos faltantes
# 2. Dtype: ¿los tipos son correctos? (una columna numérica que aparece como 'object' es sospechosa)
# 3. Memory usage: ¿el dataset cabe cómodamente en memoria?

print("\n--- Tipos de datos ---")
print(df_rendimiento.dtypes)

print("\n--- Valores nulos por columna ---")
print(df_rendimiento.isnull().sum())

print("\n--- Porcentaje de nulos ---")
print((df_rendimiento.isnull().sum() / len(df_rendimiento) * 100).round(1))

---

## 3. Ordenamiento de datos

Ordenar datos es fundamental para exploración y para detectar patrones.

In [ ]:
# Crear dataset de ejemplo
df_ejemplo = pd.DataFrame({
    'estudiante': ['Ana', 'Carlos', 'María', 'José', 'Laura', 'Pedro'],
    'carrera': ['Administración', 'Contaduría', 'Economía', 'Administración', 'Contaduría', 'Economía'],
    'nota_final': [16, 12, 8, 14, 19, 11],
    'semestre': [4, 6, 3, 8, 5, 7]
})

# Ordenar por una columna
df_ejemplo.sort_values('nota_final', ascending=False)

In [ ]:
# Ordenar por múltiples columnas
df_ejemplo.sort_values(['carrera', 'nota_final'], ascending=[True, False])

In [ ]:
# sort_index: útil después de operaciones que desordenan el índice
df_filtrado = df_ejemplo[df_ejemplo['nota_final'] >= 12]
print("Índice desordenado:")
print(df_filtrado.index.tolist())

df_filtrado_ordenado = df_filtrado.sort_index()
print("\nÍndice ordenado:")
print(df_filtrado_ordenado.index.tolist())

---

## 4. Creación de variables derivadas

Transformar datos existentes en nuevas variables que aporten más información al análisis.

In [ ]:
# Dataset de nómina
df_nom = pd.DataFrame({
    'empleado': ['García', 'López', 'Martínez', 'Rodríguez', 'Hernández'],
    'departamento': ['Ventas', 'Admin', 'Ventas', 'IT', 'Admin'],
    'salario_base': [1200, 950, 1350, 1800, 1100],
    'bono': [200, 100, 300, 250, 150],
    'antiguedad_anios': [5, 2, 8, 3, 12]
})

# Variable derivada: salario total
df_nom['salario_total'] = df_nom['salario_base'] + df_nom['bono']

# Variable derivada: porcentaje del bono
df_nom['bono_pct'] = (df_nom['bono'] / df_nom['salario_base'] * 100).round(1)

# Variable derivada categórica: rango de antigüedad
df_nom['rango_antiguedad'] = pd.cut(
    df_nom['antiguedad_anios'],
    bins=[0, 3, 7, 100],
    labels=['Junior', 'Mid', 'Senior']
)

df_nom

In [ ]:
# Variables derivadas con condiciones
df_nom['requiere_aumento'] = np.where(
    (df_nom['antiguedad_anios'] > 5) & (df_nom['salario_base'] < 1200),
    'Sí',
    'No'
)

df_nom[['empleado', 'salario_base', 'antiguedad_anios', 'requiere_aumento']]

---

## 5. Tratamiento de datos faltantes

Los datos faltantes no son solo un problema técnico — cada decisión tiene impacto en el análisis.

### La pregunta clave: ¿por qué falta este dato?

| Tipo de faltante | Ejemplo | Estrategia |
|-----------------|---------|------------|
| **Aleatorio** | Nota no registrada por error administrativo | Imputar (media, mediana) |
| **Sistemático** | Estudiantes que abandonaron no tienen nota | No imputar — el faltante ES información |
| **Estructural** | Campo "empresa" vacío para estudiantes sin pasantía | Rellenar con valor descriptivo ("Sin pasantía") |

In [ ]:
# Dataset con datos faltantes (contexto socioeconómico)
df_encuesta = pd.DataFrame({
    'estudiante_id': range(1, 11),
    'ingreso_familiar': [800, np.nan, 1200, 450, np.nan, 3500, 600, np.nan, 900, 1500],
    'horas_trabajo': [20, 40, 0, 30, np.nan, 0, 25, 35, np.nan, 10],
    'nota_promedio': [14, 11, 17, 9, np.nan, 18, 13, 10, 15, 16],
    'motivo_baja': [np.nan, np.nan, np.nan, 'Económico', 'Abandono', np.nan, np.nan, 'Laboral', np.nan, np.nan]
})

print("Datos faltantes por columna:")
print(df_encuesta.isnull().sum())
print()
df_encuesta

In [ ]:
# Estrategia 1: Eliminar filas (solo si son pocas y aleatorias)
df_sin_nulos = df_encuesta.dropna(subset=['nota_promedio'])
print(f"Filas antes: {len(df_encuesta)}, después: {len(df_sin_nulos)}")

In [ ]:
# Estrategia 2: Imputar con mediana (más robusto que la media ante outliers)
mediana_ingreso = df_encuesta['ingreso_familiar'].median()
print(f"Mediana de ingreso familiar: ${mediana_ingreso}")

df_imputado = df_encuesta.copy()
df_imputado['ingreso_familiar'] = df_imputado['ingreso_familiar'].fillna(mediana_ingreso)
df_imputado[['estudiante_id', 'ingreso_familiar']]

In [ ]:
# Estrategia 3: El faltante ES información
# motivo_baja es NaN para quienes NO se dieron de baja — eso no es un error
df_imputado['motivo_baja'] = df_imputado['motivo_baja'].fillna('Activo')
df_imputado[['estudiante_id', 'motivo_baja']]

### Reflexión sobre imputación en contexto socioeconómico

Cuando trabajas con datos de ingreso familiar, empleo o condiciones socioeconómicas:

- **No reportar ingreso puede significar** que es muy bajo (vergüenza) o muy alto (privacidad)
- **Imputar con la media puede ocultar desigualdad** — si hay 7 estudiantes con ingresos de $500 y 3 con ingresos de $5,000, la media ($1,850) no representa a nadie
- **Considerar imputar por grupo**: la mediana del ingreso de estudiantes de la misma carrera o turno puede ser más representativa

> **Regla práctica:** Si los faltantes son >30% de una columna, no imputar — documentar la limitación.

---

## 6. Detección y manejo de outliers

Un outlier no siempre es un error. Depende del contexto.

In [ ]:
# Dataset con posibles outliers
df_salarios = pd.DataFrame({
    'empleado_id': range(1, 21),
    'departamento': ['Operaciones']*8 + ['Admin']*6 + ['Gerencia']*4 + ['Especial']*2,
    'salario': [
        800, 850, 900, 750, 820, 880, 790, 0,      # Operaciones (el 0 es error)
        1100, 1200, 1050, 1300, 1150, 1250,          # Admin
        3500, 4200, 3800, 50000,                      # Gerencia (50k podría ser real)
        950, 15000                                    # Especial
    ]
})

df_salarios.describe()

In [ ]:
# Método IQR (Rango Intercuartil)
Q1 = df_salarios['salario'].quantile(0.25)
Q3 = df_salarios['salario'].quantile(0.75)
IQR = Q3 - Q1

limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Rango aceptable: [{limite_inferior}, {limite_superior}]")

outliers_iqr = df_salarios[
    (df_salarios['salario'] < limite_inferior) | 
    (df_salarios['salario'] > limite_superior)
]
print(f"\nOutliers detectados ({len(outliers_iqr)}):")
outliers_iqr

In [ ]:
# Método Z-Score
from scipy import stats

df_salarios['z_score'] = np.abs(stats.zscore(df_salarios['salario']))

outliers_zscore = df_salarios[df_salarios['z_score'] > 3]
print(f"Outliers por z-score > 3 ({len(outliers_zscore)}):")
outliers_zscore[['empleado_id', 'departamento', 'salario', 'z_score']]

### Decisiones contextuales sobre outliers

| Valor | Contexto | Decisión |
|-------|----------|----------|
| Salario = $0 | Un empleado activo no puede ganar $0 | **Error** → investigar o eliminar |
| Salario = $50,000 | Un gerente general en empresa grande | **Puede ser real** → verificar, no eliminar automáticamente |
| Nota = 25 | La escala es 0-20 | **Error** → corregir o eliminar |
| Edad = 65 | Estudiante universitario | **Inusual pero posible** → mantener |

> **Regla práctica:** Nunca eliminar outliers automáticamente. Siempre preguntar: ¿este valor es imposible, improbable o simplemente inusual?

In [ ]:
# Tratamiento contextual de outliers
df_limpio = df_salarios.copy()

# Caso 1: Salario = 0 es claramente un error → reemplazar con NaN
df_limpio.loc[df_limpio['salario'] == 0, 'salario'] = np.nan

# Caso 2: Salario de 50,000 — marcar para revisión pero no eliminar
df_limpio['requiere_revision'] = df_limpio['salario'] > limite_superior

print("Dataset limpio:")
df_limpio[['empleado_id', 'departamento', 'salario', 'requiere_revision']].to_string()

---

## Ejercicio integrador

Usando el dataset `rendimiento_academico.csv`:

1. Cárgalo e inspecciona con `.info()`
2. Identifica columnas con datos faltantes y decide una estrategia para cada una
3. Crea al menos 2 variables derivadas que aporten al análisis
4. Detecta outliers en las columnas numéricas
5. Documenta cada decisión de limpieza

In [ ]:
# Tu código aquí
df_rend = pd.read_csv('../datasets/universidad/rendimiento_academico.csv')


---

## Resumen

| Operación | Función/Método | Cuándo usarlo |
|-----------|---------------|---------------|
| Leer Excel | `pd.read_excel()` | Datos de fuentes institucionales |
| Inspeccionar | `.info()`, `.describe()` | Siempre, como primer paso |
| Ordenar | `sort_values()`, `sort_index()` | Exploración y presentación |
| Variables derivadas | `df['nueva'] = ...` | Enriquecer el análisis |
| Faltantes | `fillna()`, `dropna()` | Después de entender el porqué |
| Outliers | IQR, z-score | Calidad de datos, no automáticamente |

**Siguiente:** Entregable — Dataset limpio y documentado